# FinIA-Flex — Paso 6: Fine-Tuning Ligero (LoRA) del Formato de Reporte

**Proyecto:** FinIA-Flex — Copiloto Financiero y de Control de Costos para manufactura
**Contexto académico:** Caso Práctico Unidad 1, materia Generative IA — Maestría en Ciencia
de Datos y Analítica Visual, Instituto Europeo de Posgrado

---

## Objetivo del notebook

Cumplir el requisito 5 del caso práctico ("ajustar el LLM en un conjunto de datos de alta
calidad para mejorar su desempeño") mediante un ajuste fino ligero (LoRA) sobre un modelo
pequeño, usando el dataset sintético de 70 ejemplos generado en este paso.

**Qué se ajusta y qué no:** este fine-tuning no enseña al modelo hechos financieros nuevos
(eso ya lo cubre el RAG, Pasos 3 y 5) ni le enseña a hacer cálculos (eso ya lo cubre el
cálculo determinístico en Python, Paso 5). Lo que se ajusta es el **estilo y la estructura**
de redacción: que el modelo produzca, de forma consistente y sin necesidad de un prompt
extenso, reportes con las 5 secciones exactas en el tono y formato de FlexParts Manufacturing
MX. Esta separación de responsabilidades (RAG = hechos, código = cálculo, fine-tuning =
estilo) es una decisión de arquitectura deliberada, no una limitación del prototipo.

## Por qué LoRA y no fine-tuning completo

Ajustar todos los parámetros de un LLM completo requiere GPUs de alto costo y grandes
volúmenes de datos. **LoRA (Low-Rank Adaptation)** ajusta solo una fracción pequeña de los
parámetros (adaptadores), manteniendo el modelo base congelado. Esto permite entrenar en la
GPU gratuita de Google Colab (T4) en pocos minutos, con un dataset de decenas de ejemplos en
lugar de miles.

## Modelo base utilizado

Se utiliza **Llama 3.2 1B Instruct**, un modelo pequeño (1,000 millones de parámetros) que
corre cómodamente en la GPU T4 gratuita de Colab. Se usa la librería **Unsloth**, que
optimiza el entrenamiento LoRA para que sea significativamente más rápido y consuma menos
memoria que una implementación estándar de Hugging Face.


## 1. Configuración del entorno

**Importante:** antes de ejecutar, verificar que el entorno de ejecución de Colab tenga GPU
activada: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.


In [ ]:
!pip install -q unsloth

import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Cargar el modelo base con Unsloth

Se carga en 4-bit para reducir el uso de memoria de la GPU, lo cual es especialmente
importante en la GPU gratuita de Colab.


In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # detección automática según la GPU
    load_in_4bit=True,
)

print("Modelo base cargado.")

## 3. Configurar los adaptadores LoRA

**Decisión técnica:** `r=16` (rango del adaptador) y `lora_alpha=16` son valores estándar
recomendados por Unsloth para tareas de ajuste de estilo/formato en modelos pequeños — un
rango mayor no aporta beneficio significativo para una tarea de esta escala (70 ejemplos) y
solo incrementaría el tiempo de entrenamiento.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Adaptadores LoRA configurados.")

## 4. Cargar y preparar el dataset

Se sube `finia_flex_finetuning_dataset.jsonl` (generado en este paso, 70 ejemplos) directo al
panel de Archivos de Colab, o se coloca en la carpeta de Drive del proyecto.


In [ ]:
from datasets import load_dataset

# Ajustar la ruta según dónde se haya colocado el archivo
DATASET_PATH = "/content/finia_flex_finetuning_dataset.jsonl"
# Alternativa si se colocó en Google Drive:
# DATASET_PATH = "/content/drive/MyDrive/FinIA-Flex/finia_flex_finetuning_dataset.jsonl"

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Ejemplos cargados: {len(dataset)}")
print(dataset[0])

In [ ]:
def formatear_ejemplo(ejemplo):
    texto = tokenizer.apply_chat_template(
        ejemplo["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": texto}

dataset_formateado = dataset.map(formatear_ejemplo)
print("--- Ejemplo formateado con la plantilla de chat del modelo ---")
print(dataset_formateado[0]["text"][:800])

## 5. Entrenamiento

Con 70 ejemplos y una GPU T4, el entrenamiento toma aproximadamente 3-8 minutos. Se usan
pocas épocas (3) porque el objetivo es ajustar estilo, no memorizar contenido — más épocas
con un dataset tan pequeño arriesgaría sobreajuste (que el modelo memorice los ejemplos en
vez de generalizar el formato).

**Nota de compatibilidad:** la versión inicial de este notebook usaba `SFTTrainer` de la
librería `trl`. Esa librería cambia su API con frecuencia (dos incompatibilidades distintas
aparecieron en pruebas consecutivas: primero el parámetro `tokenizer` renombrado a
`processing_class`, después un parámetro interno `push_to_hub_token` ya no soportado). Para
evitar depender de una API inestable, se reemplazó `SFTTrainer` por el `Trainer` genérico de
`transformers` — una interfaz mucho más estable, ya que `unsloth` solo necesita el modelo con
los adaptadores LoRA ya configurados (Sección 3); el resto es un ciclo de entrenamiento
estándar de `transformers`, sin ninguna dependencia de `trl`.


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenizar_ejemplo(ejemplo):
    return tokenizer(ejemplo["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

dataset_tokenizado = dataset_formateado.map(
    tokenizar_ejemplo,
    remove_columns=dataset_formateado.column_names,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="/content/finia_flex_lora_output",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado,
    data_collator=data_collator,
)

resultado_entrenamiento = trainer.train()
print("Entrenamiento completo.")
print(resultado_entrenamiento)

## 6. Guardar el adaptador LoRA entrenado

Se guarda solo el adaptador (unos pocos megabytes), no el modelo completo — esta es una de
las ventajas de LoRA: el resultado del ajuste es pequeño y portable.


In [ ]:
RUTA_ADAPTADOR = "/content/drive/MyDrive/FinIA-Flex/finia_flex_lora_adapter"

from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained(RUTA_ADAPTADOR)
tokenizer.save_pretrained(RUTA_ADAPTADOR)

print(f"Adaptador LoRA guardado en: {RUTA_ADAPTADOR}")

## 7. Comparación antes / después (evidencia principal del Paso 6)

Se compara la respuesta del modelo **base** (sin ajustar) contra el modelo **ajustado**
(con el adaptador LoRA activo), ante el mismo caso de prueba. Esta comparación es la
evidencia central que demuestra el efecto del fine-tuning.


In [ ]:
CASO_PRUEBA = """DATOS:
Centro de costo: Línea de Producción 2
Categoría: Energía
Presupuesto: $88,000 MXN | Real: $97,500 MXN (Octubre)
Variación: 10.8% ($9,500 MXN)
Histórico: +2.1%, +4.3%
Responsable: Gerente de Línea 2 - M. Torres

INDICADORES CALCULADOS POR EL SISTEMA (usa estos valores tal cual):
- Clasificación de la variación: Variación significativa
- Umbral de aprobación de la categoría "Energía": $30,000 MXN
- ¿La variación de este mes excede el umbral de su categoría?: NO
- Meses consecutivos de sobrecosto (incluyendo el actual): 3
- ¿Aplica la regla de variación sostenida (3+ meses consecutivos de sobrecosto)?: SÍ"""

mensajes_prueba = [
    {"role": "system", "content": (
        "Eres un analista financiero senior de FlexParts Manufacturing MX, especializado en "
        "control de costos de manufactura. Redactas reportes ejecutivos de variación "
        "presupuestal para Gerencia, siguiendo siempre la estructura de 5 secciones: Resumen "
        "Ejecutivo, Diagnóstico por Centro de Costo, Alertas de Política, Recomendación, y "
        "Responsable y Siguiente Paso. Usas únicamente los datos e indicadores que se te "
        "proporcionan, sin inventar cifras, políticas ni responsables."
    )},
    {"role": "user", "content": CASO_PRUEBA},
]

inputs = tokenizer.apply_chat_template(
    mensajes_prueba, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

FastLanguageModel.for_inference(model)  # activa el adaptador LoRA para inferencia rápida
salida_ajustado = model.generate(input_ids=inputs, max_new_tokens=400, temperature=0.3, do_sample=True)
texto_ajustado = tokenizer.decode(salida_ajustado[0][inputs.shape[1]:], skip_special_tokens=True)

print("=" * 90)
print("RESPUESTA DEL MODELO AJUSTADO (con LoRA)")
print("=" * 90)
print(texto_ajustado)

In [ ]:
# Para comparar contra el modelo base, se recarga sin el adaptador LoRA
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_base)

inputs_base = tokenizer_base.apply_chat_template(
    mensajes_prueba, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

salida_base = model_base.generate(input_ids=inputs_base, max_new_tokens=400, temperature=0.3, do_sample=True)
texto_base = tokenizer_base.decode(salida_base[0][inputs_base.shape[1]:], skip_special_tokens=True)

print("=" * 90)
print("RESPUESTA DEL MODELO BASE (sin ajustar)")
print("=" * 90)
print(texto_base)

## 8. Qué revisar en la comparación

Al comparar las dos salidas de la Sección 7, documentar en el informe:

- ¿El modelo ajustado sigue la estructura de 5 secciones de forma más consistente que el
  modelo base, sin necesidad de que el prompt se lo detalle exhaustivamente?
- ¿El modelo ajustado usa el mismo tono y las mismas frases tipo ("Se activa la regla...",
  "Responsable y Siguiente Paso") que aparecían en el dataset de entrenamiento?
- ¿El modelo base, al no haber visto el formato, tiende a estructurar la respuesta de forma
  más libre o genérica?

Con modelos de 1B de parámetros y un dataset de 70 ejemplos, es normal que la mejora sea
perceptible en **consistencia de formato**, no necesariamente en calidad de razonamiento
financiero — para el razonamiento y los hechos, el prototipo sigue apoyándose en el RAG
(Paso 5) y el cálculo determinístico, no en este modelo ajustado. El fine-tuning aquí cumple
un rol de estilo, no de motor principal de decisión del prototipo.


---
## Resumen técnico (Paso 6)

**Proceso realizado:** ajuste fino ligero (LoRA, `r=16`) de Llama 3.2 1B Instruct usando
Unsloth sobre GPU T4 gratuita de Google Colab, con un dataset sintético de 70 ejemplos
generados de forma determinística (Sección previa a este notebook), cubriendo variedad de
centros de costo, categorías y tipos de alerta (umbral excedido, variación sostenida, sin
alertas).

**Decisiones técnicas documentadas:**
- Modelo pequeño (1B parámetros) elegido para viabilidad en GPU gratuita, no por ser el
  modelo principal del prototipo (ese rol lo cumple Llama 3.3 70B vía Groq, Paso 5).
- LoRA en lugar de fine-tuning completo, por costo computacional y porque el objetivo es
  ajustar estilo, no conocimiento.
- 3 épocas de entrenamiento, para evitar sobreajuste dado el tamaño reducido del dataset.
- Separación explícita de responsabilidades: RAG aporta hechos, código aporta cálculo exacto,
  fine-tuning aporta consistencia de estilo — ninguno de los tres sustituye a los otros dos.

**Hallazgos reales y corrección aplicada:** al inicializar el entrenamiento con `SFTTrainer`
de la librería `trl`, aparecieron dos errores de compatibilidad consecutivos entre versiones
de `transformers` y `trl` (primero con el parámetro `tokenizer`/`processing_class`, después
con un parámetro interno `push_to_hub_token` ya no soportado). En lugar de seguir ajustando
versiones de una librería con cambios frecuentes de API, se reemplazó `SFTTrainer` por el
`Trainer` genérico de `transformers`, eliminando por completo la dependencia de `trl` para el
entrenamiento. Esta decisión prioriza estabilidad a largo plazo sobre las últimas
funcionalidades de conveniencia que ofrece `trl`.

**Evidencia generada:** la comparación de la Sección 7 (modelo base vs. modelo ajustado) ante
el mismo caso de prueba es la evidencia principal de este paso para el informe de desarrollo.

**Siguiente paso:** Paso 7 — formalizar los mecanismos de filtrado y control de calidad
(requisito 7), construyendo sobre los tres hallazgos ya documentados en los Pasos 4 y 5.
